# Building a Multi-agent Recruiter using the Letta Framework

## Preparation

<p> 🔧 &nbsp; <b>Install Dependencies:</b> Open a terminal and navigate to the directory containing the <code>requirements.txt</code> file. Then run the following command to install the necessary libraries:</p>
<pre><code>pip install -r requirements.txt</code></pre>

<p> 🔑 &nbsp; <b>Configure OpenAI API Key:</b> This notebook requires an OpenAI API key to function. The key is managed through the <code>helper.py</code> file. 
  <ol>
    <li>Open the <code>helper.py</code> file.</li>
    <li>Locate the function <code>get_openai_api_key()</code>.</li>
    <li>You can set your API key in one of two ways:</li>
    <ul>
      <li><b>Environment Variable (Recommended):</b> Set the <code>OPENAI_API_KEY</code> environment variable in your system. The script will automatically pick it up.</li>
      <li><b>Directly in <code>helper.py</code> (Not Recommended for production):</b> If you prefer, you can hardcode your API key by modifying the line <code>api_key = "YOUR_API_KEY"</code> within the <code>get_openai_api_key</code> function. Replace <code>"YOUR_API_KEY"</code> with your actual OpenAI API key. <b>Be careful not to share your notebook or commit this file to a public repository if you choose this method.</b></li>
    </ul>
    <li>Ensure the <code>helper.py</code> file is in the same directory as this notebook or accessible in your Python path.</li>
  </ol>
</p>

</div>

## Section 0: Setup a Letta client

In [ ]:
from letta_client import Letta

client = Letta(base_url="http://localhost:8283")

In [ ]:
def print_message(message):
    if message.message_type == "reasoning_message":
        print("🧠 Reasoning: " + message.reasoning)
    elif message.message_type == "assistant_message":
        print("🤖 Agent: " + message.content)
    elif message.message_type == "tool_call_message":
        print("🔧 Tool Call: " + message.tool_call.name +  \
              "\n" + message.tool_call.arguments)
    elif message.message_type == "tool_return_message":
        print("🔧 Tool Return: " + message.tool_return)
    elif message.message_type == "user_message":
        print("👤 User Message: " + message.content)
    elif message.message_type == "system_message":
        print(" System Message: " + message.content)
    elif message.message_type == "usage_statistics":
        # for streaming specifically, we send the final
        # chunk that contains the usage statistics
        print(f"Usage: [{message}]")
        return
    print("-----------------------------------------------------")

## Section 1: Shared Memory Block

### Creating a shared memory block

In [ ]:
company_description = "The company is called AgentOS " \
+ "and is building AI tools to make it easier to create " \
+ "and deploy LLM agents."

company_block = client.blocks.create(
    value=company_description,
    label="company",
    limit=10000 # character limit
)

In [ ]:
company_block

## Section 2: Orchestrating Multiple Agents

### Creating tools for the outreach agent

In [ ]:
def draft_candidate_email(content: str):
    """
    Draft an email to reach out to a candidate.

    Args:
        content (str): Content of the email
    """
    return f"Here is a draft email: {content}"
draft_email_tool = client.tools.upsert_from_function(func=draft_candidate_email)

### Creating the outreach agent

In [ ]:
outreach_persona = (
    "You are responsible for drafting emails "
    "on behalf of a company with the draft_candidate_email tool. "
    "Candidates to email will be messaged to you. "
)

outreach_agent = client.agents.create(
    name="outreach_agent",
    memory_blocks=[
        {"label": "persona", "value": outreach_persona}
    ],
    model="openai/gpt-4o-mini-2024-07-18",
    embedding="openai/text-embedding-ada-002",
    tools=[draft_email_tool.name],
    block_ids=[company_block.id]
)

### Creating tools for the evaluation agent

In [ ]:
def reject(candidate_name: str): 
    """ 
    Reject a candidate. 

    Args: 
        candidate_name (str): The name of the candidate
    """
    return


reject_tool = client.tools.upsert_from_function(func=reject)

### Creating a persona for the evaluation agent

In [ ]:
skills = "Front-end (React, Typescript) or software engineering skills"

eval_persona = (
    f"You are responsible for evaluating candidates. "
    f"Ideal candidates have skills: {skills}. "
    "Reject bad candidates with your reject tool. "
    f"Send strong candidates to agent ID {outreach_agent.id}. "
    "You must either reject or send candidates to the other agent. "
)

### Creating the evaluation agent

In [ ]:
eval_agent = client.agents.create(
    name="eval_agent",
    memory_blocks=[
        {"label": "persona", "value": eval_persona}
    ],
    model="openai/gpt-4o-mini-2024-07-18",
    embedding="openai/text-embedding-ada-002",
    tool_ids=[reject_tool.id],
    tools=['send_message_to_agent_and_wait_for_reply'],
    include_base_tools=False,
    block_ids=[company_block.id],
    tool_rules = [
        {
            "type": "exit_loop",
            "tool_name": "send_message_to_agent_and_wait_for_reply"
        }
    ]
)

In [ ]:
[tool.name for tool in eval_agent.tools]

### Sending resume data to agents

In [ ]:
resume = open("resumes/tony_stark.txt", "r").read()

In [ ]:
response = client.agents.messages.create_stream(
    agent_id=eval_agent.id,
    messages=[
        {
            "role": "user",
            "content": f"Evaluate: {resume}"
        }
    ]
)
for message in response:
    print_message(message)

### Viewing outreach agent messages

In [ ]:
# print messages for `outreach_agent`
for message in client.agents.messages.list(agent_id=outreach_agent.id)[1:]: 
    print_message(message)

## Section 3: Shared Memory

### Updating information to shared memory blocks

In [ ]:
response = client.agents.messages.create_stream(
    agent_id=outreach_agent.id,
    messages=[
        {
            "role": "user",
            "content": "The company has rebranded to Letta"
        }
    ]
)
for message in response:
    print_message(message)

In [ ]:
client.agents.blocks.retrieve(
    agent_id=eval_agent.id, 
    block_label="company"
)

In [ ]:
client.agents.blocks.retrieve(
    agent_id=outreach_agent.id, 
    block_label="company"
)

## Section 4: Multi-agent groups

In [ ]:
def print_message_multiagent(message):  
    if message.message_type == "reasoning_message": 
        print(f"🧠 Reasoning ({message.name}): " + message.reasoning) 
    elif message.message_type == "assistant_message": 
        print(f"🤖 Agent ({message.name}): " + message.content) 
    elif message.message_type == "tool_call_message": 
        print(f"🔧 Tool Call ({message.name}): " + message.tool_call.name + "\n" + message.tool_call.arguments)
    elif message.message_type == "tool_return_message": 
        print(f"🔧 Tool Return ({message.name}): " + message.tool_return)
    elif message.message_type == "user_message": 
        print("👤 User Message: " + message.content)
    elif message.message_type == "usage_statistics": 
        # for streaming specifically, we send the final chunk that contains the usage statistics 
        print(f"Usage: [{message}]")
        return 
    print("-----------------------------------------------------")

### Recreating the outreach and evaluation agents

In [ ]:
# create the outreach agent 
outreach_agent = client.agents.create(
    name="outreach_agent",
    memory_blocks=[
        { "label": "persona", "value": outreach_persona}
    ],
    model="openai/gpt-4o-mini-2024-07-18",
    embedding="openai/text-embedding-ada-002",
    tool_ids=[draft_email_tool.id], 
    block_ids=[company_block.id]
)

# create the evaluation agent 
eval_agent = client.agents.create(
    name="eval_agent",
    memory_blocks=[
        { "label": "persona", "value": eval_persona}
    ],
    model="openai/gpt-4o-mini-2024-07-18",
    embedding="openai/text-embedding-ada-002",
    tool_ids=[reject_tool.id],
    block_ids=[company_block.id]
)

### Creating a round-robin agent group

In [ ]:
"""
Round-Robin Group
"""
round_robin_group = client.groups.create(
    description="This team is responsible for recruiting candidates.",
    agent_ids=[eval_agent.id, outreach_agent.id],
)

### Messaging an agent group

In [ ]:
resume = open("resumes/spongebob_squarepants.txt", "r").read()

In [ ]:
response_stream = client.groups.messages.create_stream(
    group_id=round_robin_group.id,
    messages=[
       {"role": "user", "content": f"Evaluate: {resume}"}
    ]
)

In [ ]:
for message in response_stream: 
    print_message_multiagent(message)